# Laptop Recommendation Dashboard

An interactive **Dash** app that recommends a laptop model from hardware specs and budget, backed by a `RandomForestClassifier` trained on **synthetically generated** data.

**Note on the data:** there is no real laptop-market dataset here — `generate_synthetic_data()` builds 1,000 rows from simple rule-based logic (budget bracket → laptop model, with noise). This is a from-scratch demo of the full pipeline (synthetic data → model → interactive UI), not a real recommendation engine trained on market data.

**Result:** 98% test accuracy / 99% cross-validated accuracy — expected to be high, since the synthetic labels are generated from a simple deterministic rule the model only has to recover.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrhmanhesham1/applied-ml-notebooks/blob/main/projects/laptop-recommendation-dashboard/laptop_recommendation_dashboard.ipynb)


## 1. Generate Synthetic Data & Train

Rule: budget > $2000 → MacBook Air M2, > $1500 → Dell XPS 13, ≥16GB RAM → Lenovo ThinkPad X1, else Acer Swift 3 (plus noise on budget/battery so it isn't a perfectly clean rule).

In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Function to generate synthetic data with less noise
def generate_synthetic_data(num_samples):
    processors = np.random.randint(1, 6, num_samples)
    rams = np.random.choice([4, 8, 16, 32, 64], num_samples)
    storages = np.random.choice([128, 256, 512, 1000, 2000], num_samples)
    batteries = np.random.randint(2, 21, num_samples)
    budgets = np.random.randint(500, 3001, num_samples)

    # Laptop assignment
    laptops = np.where(
        (budgets > 2000), "MacBook Air M2",
        np.where(
            (budgets > 1500), "Dell XPS 13",
            np.where(
                (rams >= 16), "Lenovo ThinkPad X1", "Acer Swift 3"
            )
        )
    )

    # Add minimal noise to Budget and Battery
    budgets = budgets + np.random.randint(-20, 20, size=num_samples)
    batteries = batteries + np.random.randint(-1, 1, size=num_samples)

    return pd.DataFrame({
        'Processor': processors,
        'RAM': rams,
        'Storage': storages,
        'Battery': batteries,
        'Budget': budgets,
        'Laptop': laptops
    })

# Generate synthetic data
df = generate_synthetic_data(1000)

# Map laptop names to numerical labels for model training
laptop_mapping = {name: i for i, name in enumerate(df['Laptop'].unique())}
df['Laptop'] = df['Laptop'].map(laptop_mapping)

# Split data into features and target
X = df.drop(columns=['Laptop'])
y = df['Laptop']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Random Forest model with tuned hyperparameters
model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Cross-Validation Accuracy: {np.mean(cv_scores):.2f} (±{np.std(cv_scores):.2f})")

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=list(laptop_mapping.keys())))

# Save the model
joblib.dump(model, 'laptop_recommendation_model.pkl')

# Load the model (for deployment)
model = joblib.load('laptop_recommendation_model.pkl')

# Reverse mapping for laptop names
reverse_mapping = {i: name for name, i in laptop_mapping.items()}

# Function to predict laptop
def predict_laptop(processor, ram, storage, battery, budget):
    input_data = np.array([[processor, ram, storage, battery, budget]])
    prediction = model.predict(input_data)[0]
    return reverse_mapping[prediction]



## 2. Interactive Dashboard

Sliders for processor tier, RAM, storage, battery life, and budget; the model predicts a recommended laptop in real time.

In [ ]:
# Build the Dash app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row(dbc.Col(html.H1("Laptop Recommendation Dashboard"), width=12)),
    dbc.Row([
        dbc.Col([
            html.H4("Hardware Specifications"),
            html.Label("Processor Type (1: Basic, 5: High-End)"),
            dcc.Slider(id='processor', min=1, max=5, step=1, value=3, marks={i: str(i) for i in range(1, 6)}),

            html.Label("RAM (GB)"),
            dcc.Slider(id='ram', min=4, max=64, step=4, value=16, marks={i: str(i) for i in range(4, 65, 8)}),

            html.Label("Storage (GB)"),
            dcc.Slider(id='storage', min=128, max=2000, step=128, value=512, marks={i: str(i) for i in range(128, 2001, 256)}),

            html.Label("Battery Life (Hours)"),
            dcc.Slider(id='battery', min=2, max=20, step=2, value=10, marks={i: str(i) for i in range(2, 21, 4)}),
        ], width=6),
        dbc.Col([
            html.H4("Budget"),
            html.Label("Budget ($)"),
            dcc.Slider(id='budget', min=500, max=3000, step=250, value=1500, marks={i: str(i) for i in range(500, 3001, 500)}),

            html.H3("Recommended Laptop:"),
            html.Div(id='laptop-output', style={'color': 'green', 'fontSize': 24, 'marginTop': 20})
        ], width=6)
    ])
])

# Callback to update the recommended laptop
@app.callback(
    Output('laptop-output', 'children'),
    [Input('processor', 'value'),
     Input('ram', 'value'),
     Input('storage', 'value'),
     Input('battery', 'value'),
     Input('budget', 'value')]
)
def update_laptop(processor, ram, storage, battery, budget):
    return predict_laptop(processor, ram, storage, battery, budget)

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

## Future Improvements

- Replace synthetic data with a real scraped laptop-listing dataset for an actually useful recommender
- Add a confidence score / top-3 alternatives instead of a single point prediction
- Persist the trained model artifact (`laptop_recommendation_model.pkl`) as a repo release asset instead of regenerating on every run
